# Module-10: Guided Lab

In [ ]:
# Install TensorFlow (deep learning library for building and training neural networks)
!pip install tensorflow

# Install Keras (high-level API that makes building neural networks easier, often used on top of TensorFlow)
!pip install keras

# Install Hugging Face Transformers (for using pre-trained NLP models like BERT, GPT, T5, BART, etc.)
!pip install transformers

This code shows how to build a simple sequence-to-sequence (Seq2Seq) model using LSTM layers to translate short English sentences into French. It walks through the essential steps of tokenizing text, padding sequences, building an encoder–decoder architecture, training the model on example sentence pairs, and finally generating translations by predicting one word at a time. This example introduces the core ideas behind Seq2Seq models used in machine translation and many other NLP tasks.


In [ ]:
# ----------------------------------------------------------
# SIMPLE ENGLISH → FRENCH SEQ2SEQ TRANSLATION WITH LSTM
# ----------------------------------------------------------
# In this example, we build a very small "sequence-to-sequence" (Seq2Seq)
# model that learns to translate short English sentences into French.
#
# Big picture:
# 1) Turn sentences into sequences of numbers (tokenization + padding).
# 2) Build an encoder–decoder model with LSTM layers.
# 3) Train the model on (input, output) sentence pairs.
# 4) Use the trained model to generate French sentences from new English inputs.
# ----------------------------------------------------------

import numpy as np
import tensorflow as tf

# Model, layers, and preprocessing tools from Keras.
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, LSTM, Dense
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# ----------------------------------------------------------
# 1. SAMPLE TRAINING DATA
# ----------------------------------------------------------
# Small set of example sentence pairs:
#   input_texts  = English sentences
#   target_texts = French sentences (what we want the model to output)
input_texts = [
    "Hello.",
    "How are you?",
    "What is your name?",
    "Good morning.",
    "Good night."
]

target_texts = [
    "Bonjour.",
    "Comment ça va?",
    "Quel est votre nom?",
    "Bonjour.",
    "Bonne nuit."
]

# ----------------------------------------------------------
# 2. TOKENIZE (TURN WORDS INTO INTEGERS)
# ----------------------------------------------------------
# We use a Tokenizer to:
#  - build a word → index dictionary
#  - convert each sentence into a list of integer indices

# ---- Tokenizer for input (English) sentences ----
input_tokenizer = Tokenizer()            # create a tokenizer object
input_tokenizer.fit_on_texts(input_texts)  # learn the vocabulary from input_texts
input_sequences = input_tokenizer.texts_to_sequences(input_texts)  # convert sentences to sequences of integers

# Maximum length of any input sentence (number of tokens)
input_maxlen = max(len(seq) for seq in input_sequences)

# Vocabulary size for input (add 1 for padding index 0)
input_vocab_size = len(input_tokenizer.word_index) + 1

# ---- Tokenizer for target (French) sentences ----
target_tokenizer = Tokenizer()
target_tokenizer.fit_on_texts(target_texts)
target_sequences = target_tokenizer.texts_to_sequences(target_texts)

# Maximum length of any target (French) sentence
target_maxlen = max(len(seq) for seq in target_sequences)

# Vocabulary size for target
target_vocab_size = len(target_tokenizer.word_index) + 1

# ----------------------------------------------------------
# 3. PAD SEQUENCES TO SAME LENGTH
# ----------------------------------------------------------
# Neural networks work with fixed-length inputs.
# pad_sequences:
#   - makes all sequences the same length
#   - uses 0s at the end (padding='post') for shorter sentences

input_sequences = pad_sequences(
    input_sequences,
    maxlen=input_maxlen,
    padding='post'
)

target_sequences = pad_sequences(
    target_sequences,
    maxlen=target_maxlen,
    padding='post'
)

# ----------------------------------------------------------
# 4. SPLIT TARGET INTO INPUT AND OUTPUT FOR TEACHER FORCING
# ----------------------------------------------------------
# For training the decoder, we shift the target sequence by one step:
#   target_input_sequences  = all tokens except the last
#   target_output_sequences = all tokens except the first
#
# Example:
#   target sequence:         [Bonjour . <pad>]
#   target_input_sequences:  [Bonjour .]
#   target_output_sequences: [      . <pad>]
#
# This way, at each time step, the decoder learns to predict the next word.
target_input_sequences = target_sequences[:, :-1]
target_output_sequences = target_sequences[:, 1:]

# ----------------------------------------------------------
# 5. BUILD THE SEQ2SEQ MODEL
# ----------------------------------------------------------
# We use:
#  - an encoder LSTM to read the English sentence
#  - a decoder LSTM to generate the French sentence
# Both use an embedding layer to map word indices to dense vectors.

latent_dim = 256  # size of the LSTM memory space (can be smaller for experiments)

# ----------------- ENCODER -----------------
# Input layer for the encoder: a sequence of token IDs (English sentence)
encoder_inputs = Input(shape=(input_maxlen,))

# Embedding layer: turns each token ID into a dense vector
encoder_embedding = tf.keras.layers.Embedding(
    input_dim=input_vocab_size,  # size of input vocabulary
    output_dim=latent_dim        # size of each embedding vector
)(encoder_inputs)

# LSTM layer:
#   return_state=True tells it to output the final hidden state (h) and cell state (c)
encoder_lstm = LSTM(
    latent_dim,
    return_state=True
)

# encoder_outputs is not used here; we only need the final states
encoder_outputs, state_h, state_c = encoder_lstm(encoder_embedding)

# We will pass these states to the decoder so it knows what the encoder read
encoder_states = [state_h, state_c]

# ----------------- DECODER -----------------
# Input layer for the decoder: previously generated word IDs (French sentence so far)
# shape=(None,) -> decoder can handle variable-length sequences at training time
decoder_inputs = Input(shape=(None,))

# Embedding for target language (French)
decoder_embedding = tf.keras.layers.Embedding(
    input_dim=target_vocab_size,
    output_dim=latent_dim
)(decoder_inputs)

# Decoder LSTM:
#   return_sequences=True because it outputs a prediction at each time step
#   return_state=True so we can use states later for inference
decoder_lstm = LSTM(
    latent_dim,
    return_sequences=True,
    return_state=True
)

# Connect decoder LSTM to:
#   - decoder_embedding as input
#   - encoder_states as initial hidden and cell states
decoder_outputs, _, _ = decoder_lstm(
    decoder_embedding,
    initial_state=encoder_states
)

# Final Dense layer with softmax:
#   - outputs a probability distribution over all target words at each time step
decoder_dense = Dense(
    target_vocab_size,
    activation='softmax'
)

decoder_outputs = decoder_dense(decoder_outputs)

# ----------------- FULL TRAINING MODEL -----------------
# The model takes:
#   [encoder_inputs, decoder_inputs]
# and outputs:
#   decoder_outputs (predicted French tokens for each time step)
model = Model(
    [encoder_inputs, decoder_inputs],
    decoder_outputs
)

# ----------------------------------------------------------
# 6. COMPILE THE MODEL
# ----------------------------------------------------------
# optimizer='adam'       -> widely used optimizer that adapts learning rates
# loss='sparse_categorical_crossentropy'
#   - used for multi-class classification with integer labels
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy'
)

# ----------------------------------------------------------
# 7. TRAIN THE MODEL
# ----------------------------------------------------------
# Note: This is a tiny dataset, so the model will not learn real translation,
# but it is useful to demonstrate how training works.
model.fit(
    [input_sequences, target_input_sequences],  # inputs: English + shifted French
    target_output_sequences,                   # outputs: next French token at each step
    batch_size=64,
    epochs=100,
    validation_split=0.2   # use 20% of data for validation
)

# ----------------------------------------------------------
# 8. BUILD INFERENCE MODELS (FOR ACTUAL TRANSLATION)
# ----------------------------------------------------------
# During training:
#   - we know the full target sentence, so we can feed it in at once.
# During inference (testing/translation):
#   - we must generate words one-by-one, feeding the previous word back into the decoder.

# -------- Encoder inference model --------
# For inference, the encoder simply takes an input sentence and returns the final states.
encoder_model = Model(
    encoder_inputs,
    encoder_states
)

# -------- Decoder inference model --------
# New input layers for previous decoder states (from the last time step)
decoder_state_input_h = Input(shape=(latent_dim,))
decoder_state_input_c = Input(shape=(latent_dim,))
decoder_states_inputs = [decoder_state_input_h, decoder_state_input_c]

# We reuse the same decoder LSTM and Dense layers as in training.
# However, we call decoder_lstm with decoder_states_inputs as initial_state.
decoder_outputs_inf, state_h_inf, state_c_inf = decoder_lstm(
    decoder_embedding,  # NOTE: decoder_embedding is still based on decoder_inputs
    initial_state=decoder_states_inputs
)

decoder_states_inf = [state_h_inf, state_c_inf]

# Apply the same Dense layer to get word probabilities.
decoder_outputs_inf = decoder_dense(decoder_outputs_inf)

# Build the final decoder model for inference.
# Inputs:
#   - decoder_inputs (current token)
#   - previous states (h, c)
# Outputs:
#   - predicted token probabilities
#   - updated states (h, c)
decoder_model = Model(
    [decoder_inputs] + decoder_states_inputs,
    [decoder_outputs_inf] + decoder_states_inf
)

# ----------------------------------------------------------
# 9. FUNCTION TO DECODE (TRANSLATE) A NEW SEQUENCE
# ----------------------------------------------------------
def decode_sequence(input_seq):
    """
    Translate an encoded English sequence into French using the inference models.
    We:
      1) Get encoder states from the English input.
      2) Start the decoder with a "start" token (here we use 'bonjour' as a simple choice).
      3) Repeatedly:
         - run decoder to get the next word
         - add the predicted word to the output sentence
         - stop if we reach a stop condition ('.' or max length).
    """

    # 1. Encode the input sentence to get the initial decoder states.
    states_value = encoder_model.predict(input_seq)

    # 2. Create an empty target sequence with length 1 (for the first word).
    target_seq = np.zeros((1, 1))

    # 3. Choose a starting token index for the decoder.
    # Here we assume 'bonjour' is in the target vocabulary and use it as a "start" word.
    if 'bonjour' in target_tokenizer.word_index:
        target_seq[0, 0] = target_tokenizer.word_index['bonjour']
    else:
        # If 'bonjour' is not in the vocabulary, we fall back to index 1
        # (you might want a dedicated <start> token in a real system).
        print("Warning: 'bonjour' not found in target vocabulary. Using index 1 as a fallback.")
        target_seq[0, 0] = 1

    stop_condition = False
    decoded_sentence = ''

    # 4. Generate words until we hit the stop condition.
    while not stop_condition:
        # Run the decoder for one step.
        output_tokens, h, c = decoder_model.predict(
            [target_seq] + states_value
        )

        # Choose the word with the highest probability at the last time step.
        sampled_token_index = np.argmax(output_tokens[0, -1, :])

        # If the predicted token is padding (index 0), we stop.
        if sampled_token_index == 0:
            break

        # Look up the word corresponding to the index.
        sampled_word = target_tokenizer.index_word[sampled_token_index]

        # Add the word to the decoded (output) sentence.
        decoded_sentence += ' ' + sampled_word

        # Stop if we have produced a period or hit the max number of words.
        if (sampled_word == '.' or
            len(decoded_sentence.split()) > target_maxlen):
            stop_condition = True

        # Update the target sequence: the next input is the word we just predicted.
        target_seq = np.zeros((1, 1))
        target_seq[0, 0] = sampled_token_index

        # Update states for the next loop iteration.
        states_value = [h, c]

    # Remove leading/trailing spaces and return the final translation.
    return decoded_sentence.strip()

# ----------------------------------------------------------
# 10. TEST THE MODEL ON THE TRAINING SENTENCES
# ----------------------------------------------------------
# We run the decode_sequence function on each input sentence
# to see what the model has learned to output.
for seq_index in range(len(input_texts)):
    # Extract one input sequence (English) at a time.
    input_seq = input_sequences[seq_index: seq_index + 1]

    # Translate it using the decoder.
    decoded_sentence = decode_sequence(input_seq)

    print('-')
    print('Input sentence:', input_texts[seq_index])
    print('Decoded sentence:', decoded_sentence)


This code builds a simple sequence-to-sequence (Seq2Seq) model with LSTM and an Attention layer to translate short English sentences into French. It walks through turning text into padded integer sequences, using an encoder LSTM to read the English input, a decoder LSTM with attention to focus on the most relevant encoder states at each step, training on example sentence pairs, and then running an inference loop that generates the French translation one word at a time.

In [ ]:
# ----------------------------------------------------------
# ENGLISH → FRENCH SEQ2SEQ TRANSLATION WITH ATTENTION (LSTM)
# ----------------------------------------------------------
# This example builds a small "sequence to sequence" (Seq2Seq) model
# that translates short English sentences into French.
#
# Main ideas:
# 1) Convert words to numbers (tokenization + padding).
# 2) Use an encoder LSTM to read the English sentence.
# 3) Use a decoder LSTM with an Attention layer to generate the French sentence.
# 4) Train the model on example sentence pairs, then use it to translate.
# ----------------------------------------------------------

import numpy as np
import tensorflow as tf

# Keras tools for building the model and preprocessing text.
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, LSTM, Dense, Embedding, Concatenate, TimeDistributed
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# ----------------------------------------------------------
# 1. SAMPLE TRAINING DATA
# ----------------------------------------------------------
# Small toy dataset:
#   input_texts  = English sentences (source language)
#   target_texts = French sentences (target language)
input_texts = [
    "Hello.",
    "How are you?",
    "What is your name?",
    "Good morning.",
    "Good night."
]

target_texts = [
    "Bonjour.",
    "Comment ça va?",
    "Quel est votre nom?",
    "Bonjour.",
    "Bonne nuit."
]

# ----------------------------------------------------------
# 2. TOKENIZE TEXT (MAP WORDS TO INTEGERS)
# ----------------------------------------------------------
# Tokenizer:
#  - learns a word → integer mapping from the data
#  - turns each sentence into a sequence of integers

# Tokenizer for English (input)
input_tokenizer = Tokenizer()
input_tokenizer.fit_on_texts(input_texts)                 # build vocabulary from English sentences
input_sequences = input_tokenizer.texts_to_sequences(input_texts)  # convert sentences to integer sequences

# Maximum length of any English sentence (in tokens)
input_maxlen = max(len(seq) for seq in input_sequences)

# Size of English vocabulary (add 1 for padding index 0)
input_vocab_size = len(input_tokenizer.word_index) + 1

# Tokenizer for French (target)
target_tokenizer = Tokenizer()
target_tokenizer.fit_on_texts(target_texts)
target_sequences = target_tokenizer.texts_to_sequences(target_texts)

# Maximum length of any French sentence
target_maxlen = max(len(seq) for seq in target_sequences)

# Size of French vocabulary
target_vocab_size = len(target_tokenizer.word_index) + 1

# ----------------------------------------------------------
# 3. PAD SEQUENCES TO SAME LENGTH
# ----------------------------------------------------------
# Neural networks expect input arrays with the same length.
# pad_sequences:
#   - makes all sequences the same length
#   - fills shorter sequences with 0s at the end (padding='post').

input_sequences = pad_sequences(
    input_sequences,
    maxlen=input_maxlen,
    padding='post'
)

target_sequences = pad_sequences(
    target_sequences,
    maxlen=target_maxlen,
    padding='post'
)

# ----------------------------------------------------------
# 4. SPLIT TARGET INTO INPUT AND OUTPUT (TEACHER FORCING)
# ----------------------------------------------------------
# During training, the decoder takes:
#   target_input_sequences  -> the French sentence shifted one step to the left
# and tries to predict:
#   target_output_sequences -> the French sentence shifted one step to the right
#
# Example target sequence:      [Bonjour, .]
# target_input_sequences:       [Bonjour]
# target_output_sequences:      [      .]
target_input_sequences = target_sequences[:, :-1]   # all tokens except last
target_output_sequences = target_sequences[:, 1:]   # all tokens except first

# ----------------------------------------------------------
# 5. BUILD THE SEQ2SEQ MODEL WITH ATTENTION
# ----------------------------------------------------------
latent_dim = 256  # size of the LSTM hidden state

# ----------------- ENCODER -----------------
# Encoder input: a sequence of English token IDs
encoder_inputs = Input(shape=(input_maxlen,))

# Embedding layer:
#  - turns each token ID into a dense vector of size latent_dim
encoder_embedding = Embedding(
    input_dim=input_vocab_size,
    output_dim=latent_dim
)(encoder_inputs)

# Encoder LSTM:
#  - return_sequences=True  so we get outputs at each time step
#  - return_state=True      so we get final hidden and cell states
encoder_lstm = LSTM(
    latent_dim,
    return_sequences=True,
    return_state=True
)

# encoder_outputs = sequence of hidden states for each input token
# state_h, state_c = final hidden and cell states (summarize whole sentence)
encoder_outputs, state_h, state_c = encoder_lstm(encoder_embedding)

# These states are passed to the decoder as its initial state.
encoder_states = [state_h, state_c]

# ----------------- DECODER -----------------
# Decoder input: a sequence of French token IDs (so far)
decoder_inputs = Input(shape=(None,))  # None = variable length

# Embedding for the French tokens
decoder_embedding = Embedding(
    input_dim=target_vocab_size,
    output_dim=latent_dim
)(decoder_inputs)

# Decoder LSTM:
#  - return_sequences=True  so we get hidden states for each output step
#  - return_state=True      so we can keep track of hidden and cell states
decoder_lstm = LSTM(
    latent_dim,
    return_sequences=True,
    return_state=True
)

# Initial state of the decoder is set from the encoder states
decoder_outputs, _, _ = decoder_lstm(
    decoder_embedding,
    initial_state=encoder_states
)

# ----------------- ATTENTION MECHANISM -----------------
# Attention layer:
#  - looks at all encoder_outputs (all source positions)
#  - focuses on the most relevant words for each decoder time step
attention = tf.keras.layers.Attention()

# attention_output has the same shape as decoder_outputs
# It contains the context vector for each decoder time step.
attention_output = attention([decoder_outputs, encoder_outputs])

# Concatenate decoder LSTM outputs and attention outputs along the last axis
# so that each time step has both:
#  - its own hidden state
#  - the attention context from the encoder
decoder_concat_input = Concatenate(axis=-1)([decoder_outputs, attention_output])

# ----------------- FINAL DENSE LAYER -----------------
# TimeDistributed(Dense(...)):
#  - apply the same Dense layer at each time step
#  - outputs a probability distribution over all French words
decoder_dense = TimeDistributed(
    Dense(target_vocab_size, activation='softmax')
)

decoder_outputs = decoder_dense(decoder_concat_input)

# ----------------- FULL TRAINING MODEL -----------------
# Inputs:
#   encoder_inputs (English)
#   decoder_inputs (French shifted)
# Output:
#   decoder_outputs (predicted French tokens at each time step)
model = Model(
    [encoder_inputs, decoder_inputs],
    decoder_outputs
)

# ----------------------------------------------------------
# 6. COMPILE THE MODEL
# ----------------------------------------------------------
# optimizer='adam' -> common adaptive optimizer
# loss='sparse_categorical_crossentropy'
#   -> good for multi-class prediction with integer labels
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy'
)

# ----------------------------------------------------------
# 7. TRAIN THE MODEL
# ----------------------------------------------------------
# Note: dataset is very small, so it will not learn real translation,
# but it is useful as a learning example.
model.fit(
    [input_sequences, target_input_sequences],
    target_output_sequences,
    batch_size=64,
    epochs=100,
    validation_split=0.2  # use 20 percent of data for validation
)

# ----------------------------------------------------------
# 8. BUILD INFERENCE MODELS (USED FOR ACTUAL TRANSLATION)
# ----------------------------------------------------------
# During training:
#   - we know the whole French sentence, so we feed it in at once.
# During inference (testing):
#   - we generate the output one word at a time.

# Encoder inference model:
#   - takes the English input and returns:
#       encoder_outputs (all time steps)
#       state_h, state_c (final states)
encoder_model = Model(
    encoder_inputs,
    [encoder_outputs] + encoder_states
)

# Decoder inference model:
# New input placeholders for:
#   - previous decoder states (h, c)
#   - encoder hidden states (for attention)
decoder_state_input_h = Input(shape=(latent_dim,))
decoder_state_input_c = Input(shape=(latent_dim,))
decoder_states_inputs = [decoder_state_input_h, decoder_state_input_c]

decoder_hidden_state_input = Input(shape=(input_maxlen, latent_dim))

# Reuse the same LSTM layer:
#   - but now we give it:
#       decoder_embedding (based on decoder_inputs)
#       previous states from decoder_states_inputs
decoder_outputs_inference, state_h, state_c = decoder_lstm(
    decoder_embedding,
    initial_state=decoder_states_inputs
)

# Apply attention again using the saved encoder hidden states
attention_output_inference = attention(
    [decoder_outputs_inference, decoder_hidden_state_input]
)

# Concatenate decoder outputs and attention context
decoder_concat_input_inference = Concatenate(axis=-1)(
    [decoder_outputs_inference, attention_output_inference]
)

# Apply the same Dense layer to get probabilities over vocabulary
decoder_outputs_inference = decoder_dense(decoder_concat_input_inference)

# Final decoder model for inference:
# Inputs:
#   - current decoder_inputs (last predicted French token)
#   - encoder hidden states
#   - previous decoder states (h, c)
# Outputs:
#   - next token probabilities
#   - updated decoder states (h, c)
decoder_model = Model(
    [decoder_inputs, decoder_hidden_state_input] + decoder_states_inputs,
    [decoder_outputs_inference, state_h, state_c]
)

# ----------------------------------------------------------
# 9. FUNCTION TO DECODE (TRANSLATE) A NEW ENGLISH SENTENCE
# ----------------------------------------------------------
def decode_sequence(input_seq):
    """
    Translate an encoded English sentence into French using the trained model.
    Steps:
      1) Run encoder_model to get encoder_outputs and initial states.
      2) Start the decoder with a "start" token (here we use 'bonjour').
      3) Repeatedly:
           - run decoder_model to predict next French word
           - add word to the output sentence
           - stop when we reach a period or a maximum length.
    """

    # 1. Encode the input sentence and get encoder outputs and states.
    encoder_outputs, state_h, state_c = encoder_model.predict(input_seq)
    states_value = [state_h, state_c]

    # 2. Initialize the target sequence of length 1 with the start token.
    target_seq = np.zeros((1, 1))

    # Use the word 'bonjour' as the start token if it exists.
    if 'bonjour' in target_tokenizer.word_index:
        target_seq[0, 0] = target_tokenizer.word_index['bonjour']
    else:
        # Fallback if 'bonjour' is not in the vocabulary.
        print("Warning: 'bonjour' not found in target vocabulary. Using index 1 as a fallback.")
        target_seq[0, 0] = 1  # you could define your own start token instead

    stop_condition = False
    decoded_sentence = ''

    # 3. Generate tokens one by one
    while not stop_condition:
        # Run the decoder model to get output probabilities and new states.
        output_tokens, h, c = decoder_model.predict(
            [target_seq, encoder_outputs] + states_value
        )

        # Choose the token with the highest probability.
        sampled_token_index = np.argmax(output_tokens[0, -1, :])

        # If the model predicts padding (index 0), we stop.
        if sampled_token_index == 0:
            break

        # Look up the word corresponding to the predicted index.
        sampled_word = target_tokenizer.index_word[sampled_token_index]

        # Add the word to the output sentence.
        decoded_sentence += ' ' + sampled_word

        # Stop if we hit a period or exceed maximum length of target sentence.
        if (sampled_word == '.' or
            len(decoded_sentence.split()) > target_maxlen):
            stop_condition = True

        # Update the target sequence to contain the last predicted word.
        target_seq = np.zeros((1, 1))
        target_seq[0, 0] = sampled_token_index

        # Update states for next loop iteration.
        states_value = [h, c]

    # Remove extra spaces at the beginning or end.
    return decoded_sentence.strip()

# ----------------------------------------------------------
# 10. TEST THE MODEL ON THE TRAINING SENTENCES
# ----------------------------------------------------------
# For each English input, we generate a French output and print it.
for seq_index in range(len(input_texts)):
    # Take one input sequence at a time.
    input_seq = input_sequences[seq_index: seq_index + 1]

    # Translate it.
    decoded_sentence = decode_sequence(input_seq)

    print('-')
    print('Input sentence:', input_texts[seq_index])
    print('Decoded sentence:', decoded_sentence)


This example demonstrates how to use a pre-trained T5 Transformer model to translate English text into French. The code loads the T5 model and tokenizer, converts the English input into token IDs, lets the model generate the translated output, and then decodes the result back into readable French text. This shows how modern Transformer models can perform translation tasks with only a few lines of code.

In [ ]:
# ----------------------------------------------------------
# ENGLISH → FRENCH TRANSLATION WITH T5 (TRANSFORMERS)
# ----------------------------------------------------------
# In this example, we use a pre-trained T5 model to translate
# a piece of text from English into French.
#
# Big idea:
# 1) Load a pre-trained T5 model and its tokenizer.
# 2) Turn the English text into token IDs (numbers).
# 3) Ask the model to generate the translated text.
# 4) Convert the token IDs back into readable text (French).
# ----------------------------------------------------------

# Import the T5 model and tokenizer from the transformers library.
# T5ForConditionalGeneration -> the model that can generate text (translation, summarization, etc.).
# T5Tokenizer                -> turns text into token IDs and back into text.
from transformers import T5ForConditionalGeneration, T5Tokenizer

# ----------------------------------------------------------
# 1. LOAD THE PRE-TRAINED T5 MODEL AND TOKENIZER
# ----------------------------------------------------------
# "t5-small" is a smaller, lighter version of T5.
# It is good for learning and quick experiments.
model_name = "t5-small"

# Load the T5 model. The first time you run this, it may download weights from the internet.
model = T5ForConditionalGeneration.from_pretrained(model_name)

# Load the tokenizer that matches the T5 model.
tokenizer = T5Tokenizer.from_pretrained(model_name)

# ----------------------------------------------------------
# 2. SAMPLE TEXT TO TRANSLATE
# ----------------------------------------------------------
# T5 expects the "task" to be written in the input.
# Here, the task is "translate English to French:" followed by the English text.
text = """translate English to French: Machine learning is a subset of artificial intelligence. It involves algorithms and statistical models to perform tasks without explicit instructions. Machine learning is widely used in various applications such as image recognition, natural language processing, and autonomous driving. It relies on patterns and inference instead of predefined rules."""

# ----------------------------------------------------------
# 3. TOKENIZE AND ENCODE THE TEXT
# ----------------------------------------------------------
# tokenizer.encode:
#   - turns the input string into a list of token IDs (numbers)
#   - return_tensors="pt" returns a PyTorch tensor (shape: [batch_size, sequence_length])
#   - max_length=512 sets a maximum length for the input
#   - truncation=True cuts off extra text if it is too long
inputs = tokenizer.encode(
    text,
    return_tensors="pt",
    max_length=512,
    truncation=True
)

# ----------------------------------------------------------
# 4. GENERATE THE TRANSLATION
# ----------------------------------------------------------
# model.generate:
#   - takes the encoded input
#   - produces token IDs for the translated text
#
# Important parameters:
#   max_length   -> maximum length of the output sequence (in tokens)
#   num_beams    -> beam search size (higher = better quality but slower)
#   early_stopping -> stop when all beams reach an end token
output_ids = model.generate(
    inputs,
    max_length=150,
    num_beams=4,
    early_stopping=True
)

# Decode the token IDs back into a readable string (French text).
# skip_special_tokens=True removes tokens like <pad>, </s>, etc.
translation = tokenizer.decode(output_ids[0], skip_special_tokens=True)

# ----------------------------------------------------------
# 5. PRINT THE TRANSLATION
# ----------------------------------------------------------
print("Translation:")
print(translation)


This example demonstrates how to visualize attention weights inside a pre-trained T5 Transformer model. The code loads a T5 model and tokenizer, sends a sample translation prompt through the model, extracts the model's attention matrices, and plots one of the attention heads as a heatmap. This gives students a simple, beginner-friendly way to see how Transformers focus on different parts of a sentence during translation.

In [ ]:
# ----------------------------------------------------------
# VISUALIZING ATTENTION SCORES IN A T5 TRANSFORMER MODEL
# ----------------------------------------------------------
# In this example, we:
# 1) Load a pre-trained T5 model and tokenizer.
# 2) Run the model on a sample input (translation task).
# 3) Try to extract the model's attention weights.
# 4) Visualize these attention scores as a heatmap.
#
# NOTE: This is an advanced topic. The goal here is just to give
#       you an idea of how attention can be inspected and plotted.
# ----------------------------------------------------------

# matplotlib is used for creating plots and visualizations.
import matplotlib.pyplot as plt

# seaborn builds on top of matplotlib and makes prettier statistical plots.
import seaborn as sns

# Import the T5 model and tokenizer from the transformers library.
from transformers import T5ForConditionalGeneration, T5Tokenizer

# ----------------------------------------------------------
# 1. LOAD THE PRE-TRAINED T5 MODEL AND TOKENIZER
# ----------------------------------------------------------
# "t5-small" is a smaller, faster version of the T5 model.
model_name = "t5-small"

# Load the T5 model. The first time you run this, it may download weights.
model = T5ForConditionalGeneration.from_pretrained(model_name)

# Load the tokenizer that matches the model's vocabulary.
tokenizer = T5Tokenizer.from_pretrained(model_name)

# ----------------------------------------------------------
# 2. FUNCTION TO VISUALIZE ATTENTION SCORES
# ----------------------------------------------------------
def visualize_attention(model, tokenizer, text):
    """
    Tokenize the input text, run it through the model with attention outputs enabled,
    and visualize one attention matrix as a heatmap.
    """

    # ------------------------------------------------------
    # STEP 2.1: TOKENIZE AND ENCODE THE INPUT TEXT
    # ------------------------------------------------------
    # tokenizer.encode:
    #   - converts text into a list of token IDs
    #   - return_tensors="pt" returns a PyTorch tensor
    #   - max_length and truncation keep the input from being too long
    inputs = tokenizer.encode(
        text,
        return_tensors="pt",
        max_length=512,
        truncation=True
    )

    # Usually, we put the model in evaluation mode (for PyTorch models)
    # so that layers like dropout are disabled. Uncomment this if needed:
    # model.eval()

    # ------------------------------------------------------
    # STEP 2.2: RUN THE MODEL AND REQUEST ATTENTION WEIGHTS
    # ------------------------------------------------------
    # model.generate:
    #   - generates output token IDs from the input
    #   - output_attentions=True asks the model to return attention weights
    #   - return_dict_in_generate=True makes the output a dictionary-like object
    outputs = model.generate(
        inputs,
        output_attentions=True,
        return_dict_in_generate=True
    )

    # ------------------------------------------------------
    # STEP 2.3: TRY TO ACCESS ATTENTION TENSORS
    # ------------------------------------------------------
    # Depending on the model and configuration, attentions may be stored in:
    #   - outputs.encoder_attentions
    #   - outputs.decoder_attentions
    #   - outputs.cross_attentions
    #
    # We check which one exists and use it.
    if hasattr(outputs, 'encoder_attentions') and outputs.encoder_attentions is not None:
        attentions = outputs.encoder_attentions
        print("Using encoder attentions.")
    elif hasattr(outputs, 'decoder_attentions') and outputs.decoder_attentions is not None:
        attentions = outputs.decoder_attentions
        print("Using decoder attentions.")
    elif hasattr(outputs, 'cross_attentions') and outputs.cross_attentions is not None:
        attentions = outputs.cross_attentions
        print("Using cross attentions.")
    else:
        # If none of these are found, we cannot visualize attention.
        print("Attention outputs not found in the model's generate output.")
        return

    # ------------------------------------------------------
    # STEP 2.4: PICK ONE ATTENTION MATRIX TO VISUALIZE
    # ------------------------------------------------------
    # The attentions object is usually:
    #   - a tuple where each element corresponds to one layer
    #   - each element is a tensor with shape:
    #       (batch_size, num_heads, sequence_length, sequence_length)
    #
    # For simplicity, we:
    #   - take the last layer (attentions[-1])
    #   - use batch 0 (index 0)
    #   - use head 0 (index 0)
    # Then convert it to a NumPy array for plotting.
    if isinstance(attentions, tuple) and len(attentions) > 0:
        # attentions[-1]  -> last layer
        # [0]             -> first item in batch
        # [0]             -> first attention head
        attention_matrix = attentions[-1][0][0].detach().numpy()
    else:
        print("Attention outputs are not in the expected format.")
        return

    # ------------------------------------------------------
    # STEP 2.5: PLOT THE ATTENTION MATRIX AS A HEATMAP
    # ------------------------------------------------------
    # attention_matrix is a 2D array where:
    #   rows    = positions in the output
    #   columns = positions in the input
    # Each cell shows how much one token attends to another.
    plt.figure(figsize=(10, 8))

    # sns.heatmap draws a colored grid where brighter/darker values
    # indicate higher/lower attention scores.
    sns.heatmap(attention_matrix, cmap="viridis")

    plt.title("Self-Attention Scores")
    plt.xlabel("Input Tokens")
    plt.ylabel("Output Tokens")
    plt.show()

# ----------------------------------------------------------
# 3. VISUALIZE ATTENTION FOR A SAMPLE INPUT
# ----------------------------------------------------------
# Sample text instructing T5 to translate from English to French.
sample_text = "translate English to French: How are you?"

# Before calling visualize_attention, we check that the model
# and tokenizer exist in the current Python environment.
if 'model' in locals() and 'tokenizer' in locals():
    visualize_attention(model, tokenizer, sample_text)
else:
    print("Model and tokenizer are not defined. Please ensure they are loaded.")
